In [16]:
import re
import pandas as pd


# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/df_repu.csv", low_memory=False, dtype={"ID_orateur": str}
)


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["Texte_clean"] = df["texte"].apply(nettoyer_texte)
df = df.dropna(subset=["Texte_clean"])

In [18]:
# # Exemple de regex d'exclusion Basique
# pattern_excl_case_sensitive = re.compile(r"\b[LlDd]es Républicains\b")
# pattern_excl_case_insensitive = re.compile(
#     r"\brépublique en marche\b|\bgauche démocrate et républicaine\b", re.I
# )

# pattern_lexical = re.compile(r"républi", re.I)

# =============================================
# Mais on part de la notre qui casse la tête
# =============================================

# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays
# ici pas besoin d'avoir un groupe de capture par pays mais juste global ok
pattern_pays = r"(\b(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")\b)"


# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure

# Expressions à exclure - casse exacte
pattern_excl_case_sensitive = re.compile(
    r"\b[LlDd]es Républicains\b"  # garde la casse pour identifier le parti (et pas un adjectif)
)  # voir pour élu Républicain ? doute

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"|(\bgauche démocrate et républicaine)"
    r"|(\brépublique en marche\b)"
    r"|(\bsocialiste, écologiste et républicain\b)"
    # fonctions et institutions
    r"|(\bprésident[s]? de la République\b)"
    r"|(\bprésidence[s]? de la République\b)"
    r"|(\bprocureur[s]? de la République\b)"
    r"|(\bcour[s]? de justice de la République\b)"
    r"|(\bcour[s]? de sûreté de la République\b)"
    r"|(\badministration générale de la République\b)"
    r"|(\bGouvernement de la République française\b)"
    # pays
    r"|(\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions
    r"|(" + pattern_pays + ")",  # ajout des exclusions de pays si existe
    re.I,
)

# ================================================
# On garde pour s'en rappeler mais on utilise
# la version pour l'extraction de contexte dessous
# d'ailleurs pourrait améliorer avec compréhension
# de liste etc.
# ================================================


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
    )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False


# ================================================
# Version extraction ici.
# Sans doute plus propre côté syntaxe d'ailleurs
# ================================================

import spacy

nlp = spacy.load("fr_core_news_sm")


def split_sentences_spacy(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]


# def extract_sentence_contexts_spacy(series):
#     contexts = []

#     for text in series.dropna():
#         sentences = split_sentences_spacy(text)

#         for sent in sentences:
#             excl_positions = []

#             excl_positions.extend(
#                 [m.span() for m in pattern_excl_case_sensitive.finditer(sent)]
#             )

#             excl_positions.extend(
#                 [m.span() for m in pattern_excl_case_insensitive.finditer(sent)]
#             )

#             def in_excl(pos):
#                 return any(start <= pos < end for start, end in excl_positions)

#             for match in pattern_lexical.finditer(sent):
#                 pos = match.start()

#                 if not in_excl(pos):
#                     contexts.append(sent)
#                     break

#     return contexts


def extract_sentence_contexts_spacy(series):
    contexts = []

    for doc in nlp.pipe(series.dropna(), batch_size=50):
        for sent in doc.sents:
            sent = sent.text

            excl_positions = []
            excl_positions.extend(
                [m.span() for m in pattern_excl_case_sensitive.finditer(sent)]
            )
            excl_positions.extend(
                [m.span() for m in pattern_excl_case_insensitive.finditer(sent)]
            )

            def in_excl(pos):
                return any(start <= pos < end for start, end in excl_positions)

            for match in pattern_lexical.finditer(sent):
                if not in_excl(match.start()):
                    contexts.append(sent)
                    break

    return contexts

/Users/leo/anaconda3/envs/myenv_clone/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'fr_core_news_sm' (3.5.0) was trained with spaCy v3.5.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [19]:
contexts = extract_sentence_contexts_spacy(df["Texte_clean"])

In [24]:
contexts_phrase = pd.DataFrame({"context": contexts})

In [25]:
contexts_phrase.to_csv("contexts_phrase.csv", index=False)

In [26]:
contexts_phrase

,context
0,Tout le sens d'une mixité sociale réussie est ...
1,"Nous n'avons pas peur, mais je n'oublie pas qu..."
2,Vous savez que la République est une accumulat...
3,"Voilà, donc, qui ne favorise pas la promotion ..."
4,"Madame la ministre du travail, il y a un an, n..."
...,...
18771,"remercie aussi Mme Parmentier-Lecocq, qui a fa..."
18772,Lorsqu'on affiche sur les réseaux sociaux et d...
18773,"Vous n'êtes pas un activiste d'amphi, vous ête..."
18774,"Monsieur le député, vous vous exposez, sourire..."


In [27]:
df = contexts_phrase

In [29]:
from itertools import combinations
import networkx as nx
from ipysigma import Sigma
import stopwordsiso as stopwords
import re

# stopwords français
stop_fr = stopwords.stopwords("fr")

# regex pour la racine répu*
pattern_repu = re.compile(r"répu\w*", re.IGNORECASE)


G = nx.Graph()

for _, row in df.iterrows():
    tokens = row["context"].lower().split()

    tokens = [
        t
        for t in tokens
        if t not in stop_fr  # enlever stopwords
        and not pattern_repu.match(t)  # enlever répu*
        and len(t) > 2  # enlever mots très courts
    ]

    for w1, w2 in combinations(tokens, 2):
        if G.has_edge(w1, w2):
            G[w1][w2]["weight"] += 1
        else:
            G.add_edge(w1, w2, weight=1)

edges_to_remove = [(u, v) for u, v, w in G.edges(data="weight") if w < 5]

G.remove_edges_from(edges_to_remove)

for n in G.nodes():
    G.nodes[n]["size"] = G.degree(n)
    G.nodes[n]["label"] = n

reseau = Sigma(G, node_size="size", edge_size="weight", node_label="label")
reseau


Sigma(nx.Graph with 32,646 nodes and 23,089 edges)

In [30]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))

for i, com in enumerate(communities):
    for node in com:
        G.nodes[node]["community"] = i

greedy = Sigma(
    G,
    node_size="size",
    edge_size="weight",
    node_color="community",
    node_label=lambda n: n,
)

In [31]:
greedy

Sigma(nx.Graph with 32,646 nodes and 23,089 edges)

In [ ]:
# Extraire le plus grand composant connexe
largest_cc = max(nx.connected_components(G), key=len)
G_central = G.subgraph(largest_cc).copy()

# Visualiser le réseau central
reseau_central = Sigma(
    G_central, node_size="size", edge_size="weight", node_label="label"
)
reseau_central

Sigma(nx.Graph with 3,209 nodes and 23,055 edges)

In [35]:
from networkx.algorithms.community import greedy_modularity_communities

# Appliquer sur le réseau central
communities_central = list(greedy_modularity_communities(G_central))

# Annoter les noeuds avec leur communauté
for i, com in enumerate(communities_central):
    for node in com:
        G_central.nodes[node]["community"] = i

# Visualiser
greedy_central = Sigma(
    G_central,
    node_size="size",
    edge_size="weight",
    node_color="community",
    node_label=lambda n: n,
)
greedy_central

Sigma(nx.Graph with 3,209 nodes and 23,055 edges)

In [36]:
import community  # python-louvain

# Appliquer Louvain sur le réseau central
partition = community.best_partition(G_central)

# Annoter les noeuds avec leur communauté
for node, comm in partition.items():
    G_central.nodes[node]["community"] = comm

# Visualiser avec Sigma
louvain_central = Sigma(
    G_central,
    node_size="size",
    edge_size="weight",
    node_color="community",
    node_label=lambda n: n,
)
louvain_central

ModuleNotFoundError: No module named 'community'

In [42]:
test = Sigma(
    G_central,
    node_size="size",
    edge_size="weight",
    node_metrics=["louvain"],
    node_color="louvain",
    node_size_range=(3, 20),
    default_edge_type="curve",
    node_border_color_from="node",
    # node_label_size=G.degree,
    # show_all_labels=True,
    # Des options si on veut fignoler :
    # ex pour la detection de communauté plus fine :
    # node_metrics={"community": {"name": "louvain", "resolution": 1.2}},
    # node_color="community",
    # edge_color= # mettre une couleur en fonction d'une métrique
    # label_rendered_size_threshold=10,  # ou jouer sur l'affichage labels
    # default_node_label_size=14, # mettre une taille par défaut
)

In [43]:
test

Sigma(nx.Graph with 3,209 nodes and 23,055 edges)

In [44]:
test.to_html("test_sigma.html")

Length of context_counts_5 (window=5): 19370
Length of context_counts_2 (window=2): 14941
Length of context_counts_1 (window=1): 6229
